# Extreme Space-Weather Events and the Solar Cycle
## 극한 우주기상 사건과 태양 주기

**Paper**: Owens, M.J., Lockwood, M., Barnard, L.A., Scott, C.J., Haines, C., Macneil, A. (2021)  
**Journal**: Solar Physics, 296, 82

### Overview / 개요

This notebook reproduces the core probabilistic analysis from Owens et al. (2021):

이 노트북은 Owens et al. (2021)의 핵심 확률론적 분석을 재현합니다:

1. **Synthetic $aa_H$ data generation** — generate a realistic 150-year geomagnetic record  
   합성 $aa_H$ 데이터 생성 — 150년 지자기 기록 시뮬레이션
2. **Four probability models** — Random, Phase, Phase+Amp, EarlyLate  
   네 가지 확률 모델 구현
3. **Monte Carlo simulation** — storm-occurrence time series generation  
   Monte Carlo 시뮬레이션 — 폭풍 발생 시계열 생성
4. **Statistical tests** — active/quiet ratio, cycle-amplitude correlation, odd/even asymmetry  
   통계적 검정 — 활동기/정적기 비율, 주기 진폭 상관, 홀수/짝수 비대칭
5. **Solar Cycle 25 forecast** — extreme-event probability under three scenarios  
   Solar Cycle 25 예측 — 세 가지 시나리오별 극한 사건 확률

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional
from scipy import stats

plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

# Reproducibility
RNG = np.random.default_rng(42)

## 1. Solar Cycle Data / 태양 주기 데이터

We define historical solar cycles using actual minimum dates and mean sunspot numbers.
The solar cycle phase $\phi$ is linearly normalized from 0 (minimum start) to 1 (next minimum end).

실제 태양 극소기 날짜와 평균 흑점 수를 사용하여 태양 주기를 정의합니다.
태양 주기 위상 $\phi$는 0 (극소기 시작) → 1 (다음 극소기 종료)로 선형 정규화됩니다.

$$\phi(t) = \frac{t - t_{\min,\text{start}}}{t_{\min,\text{end}} - t_{\min,\text{start}}}$$

In [ ]:
@dataclass
class SolarCycle:
    """Represents a single solar cycle with its properties.

    Attributes:
        number: Solar cycle number (e.g., 11 for SC11).
        min_year: Year of cycle start (solar minimum).
        max_year: Approximate year of solar maximum.
        mean_ssn: Mean sunspot number over the cycle.
    """
    number: int
    min_year: float
    max_year: float
    mean_ssn: float

    @property
    def is_even(self) -> bool:
        return self.number % 2 == 0

    @property
    def length(self) -> float:
        return 0  # set later from next cycle's min_year


# Historical solar cycles 11-24 (covering the aa record, 1868-2020)
# Minimum years and mean sunspot numbers from the paper's references
# (van Driel-Gesztelyi and Owens, 2020; SILSO data)
SOLAR_CYCLES = [
    SolarCycle(11, 1867.2, 1870.6, 99),
    SolarCycle(12, 1878.9, 1883.9, 64),
    SolarCycle(13, 1889.6, 1894.1, 69),
    SolarCycle(14, 1901.7, 1906.2, 45),
    SolarCycle(15, 1913.6, 1917.6, 70),
    SolarCycle(16, 1923.6, 1928.4, 55),
    SolarCycle(17, 1933.8, 1937.4, 80),
    SolarCycle(18, 1944.2, 1947.5, 100),
    SolarCycle(19, 1954.3, 1957.9, 126),
    SolarCycle(20, 1964.9, 1968.9, 81),
    SolarCycle(21, 1976.5, 1979.9, 108),
    SolarCycle(22, 1986.8, 1989.6, 101),
    SolarCycle(23, 1996.4, 2000.3, 80),
    SolarCycle(24, 2008.9, 2014.3, 57),
]

# End of record
END_YEAR = 2018.0
# SC25 start
SC25_START = 2020.0

# Mean sunspot number over entire record
MEAN_SSN = np.mean([sc.mean_ssn for sc in SOLAR_CYCLES])

print(f"Solar Cycles: {SOLAR_CYCLES[0].number}–{SOLAR_CYCLES[-1].number}")
print(f"Period: {SOLAR_CYCLES[0].min_year:.1f}–{END_YEAR:.1f}")
print(f"Mean SSN over record: {MEAN_SSN:.1f}")
print(f"Number of complete cycles: {len(SOLAR_CYCLES)}")

In [ ]:
def get_cycle_for_year(year: float) -> Optional[SolarCycle]:
    """Return the solar cycle that contains the given year.

    Args:
        year: Calendar year (fractional).

    Returns:
        SolarCycle object, or None if outside the record.
    """
    for i, sc in enumerate(SOLAR_CYCLES):
        next_min = (SOLAR_CYCLES[i + 1].min_year
                    if i + 1 < len(SOLAR_CYCLES) else SC25_START)
        if sc.min_year <= year < next_min:
            return sc
    return None


def get_phase(year: float) -> Optional[float]:
    """Compute solar-cycle phase for a given year.

    Phase is linearly normalized: 0 at cycle start (minimum),
    1 at cycle end (next minimum).

    Args:
        year: Calendar year (fractional).

    Returns:
        Phase in [0, 1], or None if outside the record.
    """
    for i, sc in enumerate(SOLAR_CYCLES):
        next_min = (SOLAR_CYCLES[i + 1].min_year
                    if i + 1 < len(SOLAR_CYCLES) else SC25_START)
        if sc.min_year <= year < next_min:
            return (year - sc.min_year) / (next_min - sc.min_year)
    return None


def is_active(phase: float) -> bool:
    """Check if the given phase falls in the active phase.

    Active phase: 0.18 <= phase <= 0.79 (centered on solar maximum).

    Args:
        phase: Solar-cycle phase in [0, 1].

    Returns:
        True if in active phase.
    """
    return 0.18 <= phase <= 0.79


def is_early_active(phase: float) -> bool:
    """Check if phase falls in the early half of the active phase.

    Early active: 0.18 <= phase < 0.485 (midpoint of active phase).
    """
    return 0.18 <= phase < 0.485


# Demonstrate with a test year
test_year = 1990.0
sc = get_cycle_for_year(test_year)
phase = get_phase(test_year)
print(f"Year {test_year}: SC{sc.number} (even={sc.is_even}), "
      f"phase={phase:.3f}, active={is_active(phase)}, "
      f"early_active={is_early_active(phase)}")

## 2. Synthetic $aa_H$ Data Generation / 합성 $aa_H$ 데이터 생성

Since the actual $aa_H$ data requires downloading from external sources, we generate
a synthetic dataset that reproduces the key statistical properties observed in the paper:

실제 $aa_H$ 데이터는 외부 소스에서 다운로드해야 하므로, 논문에서 관측된
핵심 통계적 특성을 재현하는 합성 데이터를 생성합니다:

- Right-skewed distribution (most days are quiet) / 오른쪽으로 치우친 분포 (대부분의 날은 조용함)
- Higher values during active phase / 활동기에 높은 값
- Larger storms in larger cycles / 큰 주기에서 더 큰 폭풍
- Odd/even asymmetry at extreme thresholds / 극한 임계값에서의 홀수/짝수 비대칭

In [ ]:
def generate_synthetic_aah(
    start_year: float = 1868.0,
    end_year: float = 2018.0,
    rng: np.random.Generator = RNG,
) -> tuple[np.ndarray, np.ndarray]:
    """Generate synthetic daily aa_H values mimicking real data properties.

    The synthetic data uses an exponential base distribution modulated by
    solar-cycle phase, amplitude, and odd/even asymmetry.

    Args:
        start_year: Start of the record.
        end_year: End of the record.
        rng: Random number generator.

    Returns:
        Tuple of (years, aah_values) as 1D arrays with daily resolution.
    """
    n_days = int((end_year - start_year) * 365.25)
    years = np.linspace(start_year, end_year, n_days, endpoint=False)

    # Base: exponential distribution (right-skewed, most days quiet)
    base_scale = 12.0  # median ~8 nT
    aah = rng.exponential(scale=base_scale, size=n_days)

    for i, year in enumerate(years):
        sc = get_cycle_for_year(year)
        phase = get_phase(year)
        if sc is None or phase is None:
            continue

        # Solar-cycle modulation: boost during active phase
        if is_active(phase):
            # Amplitude scaling: larger cycles → stronger boost
            amp_factor = 1.5 * sc.mean_ssn / MEAN_SSN
            aah[i] *= (1.0 + 2.5 * amp_factor)

            # Odd/even asymmetry for extreme events (adds extra boost)
            if aah[i] > 80:  # only affects already-strong events
                if sc.is_even and is_early_active(phase):
                    aah[i] *= 1.3
                elif not sc.is_even and not is_early_active(phase):
                    aah[i] *= 1.3
        else:
            # Quiet phase: slight reduction
            aah[i] *= 0.6

    return years, aah


years, aah = generate_synthetic_aah()

# Compute percentile thresholds (matching paper's approach)
percentiles = [90, 99, 99.9, 99.99]
thresholds = {p: np.percentile(aah, p) for p in percentiles}

print("Synthetic aa_H statistics:")
print(f"  Total days: {len(aah):,}")
print(f"  Mean: {aah.mean():.1f} nT")
print(f"  Median: {np.median(aah):.1f} nT")
print(f"  Max: {aah.max():.1f} nT")
print(f"\nPercentile thresholds:")
for p, t in thresholds.items():
    n_storms = np.sum(aah > t)
    print(f"  {p:>6}th: {t:>6.1f} nT  (N = {n_storms:,} storm days)")

In [ ]:
# Visualize the synthetic data — corresponds to Figure 1 in the paper
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: daily aa_H values
axes[0].plot(years, aah, color="red", alpha=0.1, lw=0.3, label="Daily $aa_H$")

# 365-day running mean
window = 365
kernel = np.ones(window) / window
aah_smooth = np.convolve(aah, kernel, mode="same")
axes[0].plot(years, aah_smooth, color="black", lw=1, label="1-year mean")
axes[0].set_ylabel("$aa_H$ [nT]")
axes[0].set_title("Synthetic $aa_H$ Record / 합성 $aa_H$ 기록 (cf. Figure 1)")
axes[0].legend()
axes[0].set_ylim(0, 200)

# Shade even-numbered cycles in gray
for sc in SOLAR_CYCLES:
    if sc.is_even:
        idx = SOLAR_CYCLES.index(sc)
        next_min = (SOLAR_CYCLES[idx + 1].min_year
                    if idx + 1 < len(SOLAR_CYCLES) else SC25_START)
        for ax in axes:
            ax.axvspan(sc.min_year, next_min, alpha=0.1, color="gray")

# Bottom: storm occurrence at different thresholds
colors = ["blue", "orange", "green", "red"]
for (p, t), color in zip(thresholds.items(), colors):
    storm_mask = aah > t
    # Compute annual storm probability (fraction of storm days per year)
    bin_edges = np.arange(1868, 2019)
    storm_prob = []
    bin_centers = []
    for y in bin_edges[:-1]:
        mask = (years >= y) & (years < y + 1)
        if mask.sum() > 0:
            storm_prob.append(storm_mask[mask].mean())
            bin_centers.append(y + 0.5)
    axes[1].semilogy(bin_centers, storm_prob,
                     color=color, alpha=0.7, lw=1,
                     label=f"{p}th centile (>{t:.0f} nT)")

axes[1].set_ylabel("Storm probability [day$^{-1}$]")
axes[1].set_xlabel("Year")
axes[1].legend(fontsize=9)
axes[1].set_ylim(1e-4, 1)

plt.tight_layout()
plt.show()

## 3. Four Probability Models / 네 가지 확률 모델

The paper constructs four hierarchical models, each adding one physical ingredient:

논문은 네 가지 계층적 모델을 구축하며, 각 모델은 하나의 물리적 요소를 추가합니다:

| Model | Description / 설명 |
|---|---|
| **Random** | Equal probability at all times (null hypothesis) / 모든 시점에서 동일한 확률 (귀무가설) |
| **Phase** | Active phase probability 9× higher than quiet / 활동기 확률이 정적기보다 9배 높음 |
| **Phase+Amp** | Phase model + cycle amplitude scaling / Phase 모델 + 주기 진폭 스케일링 |
| **EarlyLate** | Phase+Amp + odd/even early/late asymmetry / Phase+Amp + 홀수/짝수 초기/후기 비대칭 |

In [ ]:
class StormModel:
    """Base class for storm-occurrence probability models.

    Each model assigns a relative probability to each day in the record.
    Subclasses override _raw_probability() to implement different assumptions.

    Attributes:
        years: Array of calendar years (daily resolution).
        name: Model name for labeling.
    """

    def __init__(self, years: np.ndarray, name: str = "Model"):
        self.years = years
        self.name = name
        self._prob = None  # cached normalized probabilities

    def _raw_probability(self, year: float) -> float:
        """Return unnormalized relative probability for a given year."""
        raise NotImplementedError

    def probabilities(self) -> np.ndarray:
        """Return normalized probability array (sums to 1)."""
        if self._prob is None:
            raw = np.array([self._raw_probability(y) for y in self.years])
            self._prob = raw / raw.sum()
        return self._prob

    def cdf(self) -> np.ndarray:
        """Return cumulative distribution function."""
        return np.cumsum(self.probabilities())

    def generate_storms(
        self, n_storms: int, rng: np.random.Generator = RNG
    ) -> np.ndarray:
        """Generate storm times using inverse CDF sampling.

        Corresponds to the Monte Carlo method in Section 4 of the paper.

        Args:
            n_storms: Number of storms to generate.
            rng: Random number generator.

        Returns:
            Array of storm years (sorted).
        """
        cumulative = self.cdf()
        uniform_samples = rng.random(n_storms)
        # Find the year closest to each CDF value
        storm_indices = np.searchsorted(cumulative, uniform_samples)
        storm_indices = np.clip(storm_indices, 0, len(self.years) - 1)
        return np.sort(self.years[storm_indices])


class RandomModel(StormModel):
    """Null hypothesis: storms occur completely randomly."""

    def __init__(self, years: np.ndarray):
        super().__init__(years, "Random")

    def _raw_probability(self, year: float) -> float:
        return 1.0


class PhaseModel(StormModel):
    """Storm probability depends on solar-cycle phase.

    Active-phase probability is R times higher than quiet-phase probability.
    R = 9 as determined in the paper (Section 5).
    """

    def __init__(self, years: np.ndarray, active_ratio: float = 9.0):
        super().__init__(years, "Phase")
        self.active_ratio = active_ratio

    def _raw_probability(self, year: float) -> float:
        phase = get_phase(year)
        if phase is not None and is_active(phase):
            return self.active_ratio
        return 1.0


class PhaseAmpModel(StormModel):
    """Phase model + solar-cycle amplitude dependence.

    Active-phase probability is scaled by cycle amplitude:
    P_active = R * 1.5 * (A / <SSN>) * P_base

    Where A is the cycle's mean sunspot number and <SSN> is the
    record-wide mean.
    """

    def __init__(self, years: np.ndarray, active_ratio: float = 9.0,
                 amp_scale: float = 1.5):
        super().__init__(years, "Phase+Amp")
        self.active_ratio = active_ratio
        self.amp_scale = amp_scale

    def _raw_probability(self, year: float) -> float:
        phase = get_phase(year)
        sc = get_cycle_for_year(year)
        if phase is not None and sc is not None and is_active(phase):
            amp_factor = self.amp_scale * sc.mean_ssn / MEAN_SSN
            return self.active_ratio * amp_factor
        return 1.0


class EarlyLateModel(StormModel):
    """Phase+Amp model + odd/even early/late active-phase asymmetry.

    Even cycles: early active ×1.6, late active ×0.4
    Odd cycles:  early active ×0.4, late active ×1.6

    The 60% modification is set from the 99.9th percentile observations
    (Section 7 of the paper).
    """

    def __init__(self, years: np.ndarray, active_ratio: float = 9.0,
                 amp_scale: float = 1.5, asymmetry: float = 0.6):
        super().__init__(years, "EarlyLate")
        self.active_ratio = active_ratio
        self.amp_scale = amp_scale
        self.asymmetry = asymmetry

    def _raw_probability(self, year: float) -> float:
        phase = get_phase(year)
        sc = get_cycle_for_year(year)
        if phase is None or sc is None or not is_active(phase):
            return 1.0

        amp_factor = self.amp_scale * sc.mean_ssn / MEAN_SSN
        base = self.active_ratio * amp_factor

        early = is_early_active(phase)
        if sc.is_even:
            modifier = (1 + self.asymmetry) if early else (1 - self.asymmetry)
        else:
            modifier = (1 - self.asymmetry) if early else (1 + self.asymmetry)

        return base * modifier


# Instantiate all four models
models = {
    "Random": RandomModel(years),
    "Phase": PhaseModel(years),
    "Phase+Amp": PhaseAmpModel(years),
    "EarlyLate": EarlyLateModel(years),
}

print("Models initialized:")
for name, model in models.items():
    probs = model.probabilities()
    print(f"  {name:>12}: max/min prob ratio = "
          f"{probs.max() / probs[probs > 0].min():.1f}")

In [ ]:
# Visualize model probability time series — corresponds to Figure 3
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

colors_model = {"Random": "blue", "Phase": "red",
                "Phase+Amp": "black", "EarlyLate": "gray"}
styles = {"Random": "-", "Phase": "-",
          "Phase+Amp": "-", "EarlyLate": "--"}

# Top: relative probability
for name, model in models.items():
    probs = model.probabilities()
    # Smooth for visibility (annual mean)
    smoothed = np.convolve(probs, np.ones(365) / 365, mode="same")
    axes[0].plot(years, smoothed, color=colors_model[name],
                 ls=styles[name], lw=1.5, label=name, alpha=0.8)

axes[0].set_ylabel("Relative probability")
axes[0].set_title("Model Probability Time Series / "
                   "모델 확률 시계열 (cf. Figure 3)")
axes[0].legend()

# Bottom: cumulative probability
for name, model in models.items():
    axes[1].plot(years, model.cdf(), color=colors_model[name],
                 ls=styles[name], lw=1.5, label=name, alpha=0.8)

axes[1].set_ylabel("Cumulative probability")
axes[1].set_xlabel("Year")
axes[1].legend()

# Shade even cycles
for sc in SOLAR_CYCLES:
    if sc.is_even:
        idx = SOLAR_CYCLES.index(sc)
        next_min = (SOLAR_CYCLES[idx + 1].min_year
                    if idx + 1 < len(SOLAR_CYCLES) else SC25_START)
        for ax in axes:
            ax.axvspan(sc.min_year, next_min, alpha=0.08, color="gray")

plt.tight_layout()
plt.show()

## 4. Monte Carlo Simulation / Monte Carlo 시뮬레이션

For each model and threshold, we generate 5000 synthetic storm time series and compute
storm-occurrence probability during active and quiet phases. This reproduces the
methodology of Section 4 and the analysis of **Figure 4**.

각 모델과 임계값에 대해 5000개의 합성 폭풍 시계열을 생성하고, 활동기와 정적기의
폭풍 발생 확률을 계산합니다. 이는 Section 4의 방법론과 **Figure 4**의 분석을 재현합니다.

In [ ]:
def compute_phase_probabilities(
    storm_years: np.ndarray,
    total_active_days: int,
    total_quiet_days: int,
) -> tuple[float, float]:
    """Compute storm probability in active and quiet phases.

    Args:
        storm_years: Array of storm occurrence years.
        total_active_days: Number of active-phase days in the record.
        total_quiet_days: Number of quiet-phase days in the record.

    Returns:
        Tuple of (active_probability, quiet_probability) as fractions.
    """
    n_active_storms = sum(
        1 for y in storm_years
        if (p := get_phase(y)) is not None and is_active(p)
    )
    n_quiet_storms = len(storm_years) - n_active_storms

    p_active = n_active_storms / total_active_days if total_active_days > 0 else 0
    p_quiet = n_quiet_storms / total_quiet_days if total_quiet_days > 0 else 0
    return p_active, p_quiet


# Pre-compute total active/quiet days
phases = np.array([get_phase(y) for y in years])
valid_phases = phases[phases != None].astype(float)
total_active = sum(1 for p in valid_phases if is_active(p))
total_quiet = len(valid_phases) - total_active

print(f"Total days with valid phase: {len(valid_phases):,}")
print(f"Active-phase days: {total_active:,} "
      f"({100 * total_active / len(valid_phases):.1f}%)")
print(f"Quiet-phase days:  {total_quiet:,} "
      f"({100 * total_quiet / len(valid_phases):.1f}%)")

In [ ]:
def run_monte_carlo(
    model: StormModel,
    n_storms: int,
    n_iterations: int = 1000,
    rng: np.random.Generator = RNG,
) -> dict[str, np.ndarray]:
    """Run Monte Carlo simulation for a given model.

    For each iteration, generate n_storms storm times and compute
    active/quiet probabilities.

    Args:
        model: StormModel instance.
        n_storms: Number of storms per iteration.
        n_iterations: Number of Monte Carlo iterations.
        rng: Random number generator.

    Returns:
        Dict with keys 'active', 'quiet', 'diff' — each an array
        of length n_iterations.
    """
    active_probs = np.zeros(n_iterations)
    quiet_probs = np.zeros(n_iterations)

    for i in range(n_iterations):
        storms = model.generate_storms(n_storms, rng)
        p_a, p_q = compute_phase_probabilities(
            storms, total_active, total_quiet)
        active_probs[i] = p_a
        quiet_probs[i] = p_q

    return {
        "active": active_probs,
        "quiet": quiet_probs,
        "diff": active_probs - quiet_probs,
    }


# Run Monte Carlo for Random and Phase models at the 99th percentile
# (N=548 storms, 1000 iterations for speed)
n_storms_99 = int(np.sum(aah > thresholds[99]))
print(f"Running Monte Carlo for 99th percentile (N={n_storms_99} storms)...")

mc_results = {}
for model_name in ["Random", "Phase"]:
    mc_results[model_name] = run_monte_carlo(
        models[model_name], n_storms_99, n_iterations=1000)
    median_a = np.median(mc_results[model_name]["active"])
    median_q = np.median(mc_results[model_name]["quiet"])
    print(f"  {model_name:>12}: median active={median_a:.4f}, "
          f"quiet={median_q:.4f}, ratio={median_a/median_q:.1f}")

# Observed values from synthetic data
obs_storms_99 = years[aah > thresholds[99]]
obs_active, obs_quiet = compute_phase_probabilities(
    obs_storms_99, total_active, total_quiet)
print(f"\n  {'Observed':>12}: active={obs_active:.4f}, "
      f"quiet={obs_quiet:.4f}, ratio={obs_active/obs_quiet:.1f}")

In [ ]:
# Visualize Active vs Quiet comparison — corresponds to Figure 4
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

# Plot active-quiet difference for Random and Phase models
for ax_idx, (key, ylabel, title) in enumerate([
    ("quiet", "Probability [day$^{-1}$]",
     "Quiet Phase / 정적기"),
    ("active", "Probability [day$^{-1}$]",
     "Active Phase / 활동기"),
    ("diff", "Probability difference [day$^{-1}$]",
     "Active − Quiet / 활동기 − 정적기"),
]):
    ax = axes[ax_idx]

    for model_name, color in [("Random", "blue"), ("Phase", "red")]:
        data = mc_results[model_name][key]
        median = np.median(data)
        sigma1 = (np.percentile(data, 16), np.percentile(data, 84))
        sigma2 = (np.percentile(data, 2.5), np.percentile(data, 97.5))

        # Show as horizontal bands (simplified from threshold-varying plot)
        ax.axhline(median, color=color, lw=2, label=f"{model_name} median")
        ax.axhspan(sigma1[0], sigma1[1], alpha=0.2, color=color)
        ax.axhspan(sigma2[0], sigma2[1], alpha=0.1, color=color)

    # Observed value
    if key == "quiet":
        obs_val = obs_quiet
    elif key == "active":
        obs_val = obs_active
    else:
        obs_val = obs_active - obs_quiet

    ax.axhline(obs_val, color="black", lw=2, ls="--",
               label=f"Observed ({obs_val:.4f})")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=9)

axes[2].set_xlabel("(Shown at 99th percentile threshold)")

fig.suptitle("Active/Quiet Storm Probability: Observed vs Models\n"
             "활동기/정적기 폭풍 확률: 관측 vs 모델 (cf. Figure 4)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Check if Random model is rejected
random_diff = mc_results["Random"]["diff"]
obs_diff = obs_active - obs_quiet
pct_above = np.mean(random_diff >= obs_diff) * 100
print(f"\nNull hypothesis test (Random model):")
print(f"  Observed active-quiet diff: {obs_diff:.5f}")
print(f"  % of Random MC iterations with diff ≥ observed: {pct_above:.1f}%")
print(f"  → Random model {'REJECTED' if pct_above < 2.5 else 'not rejected'} "
      f"at 2σ level")

## 5. Cycle Amplitude vs Storm Occurrence / 주기 진폭 vs 폭풍 발생

Test whether bigger solar cycles produce more extreme storms by computing the
correlation between cycle-average storm probability and mean sunspot number.
This reproduces **Figures 5–6** from the paper.

태양 주기 진폭(평균 흑점 수)과 폭풍 발생 확률 사이의 상관을 계산하여
큰 태양 주기가 더 많은 극한 폭풍을 생산하는지 검증합니다.
논문의 **Figures 5–6**을 재현합니다.

In [ ]:
def cycle_storm_probability(
    threshold: float,
) -> tuple[list[float], list[float], list[int]]:
    """Compute per-cycle storm probability and mean SSN.

    Args:
        threshold: aa_H threshold for storm definition.

    Returns:
        Tuple of (mean_ssn_list, storm_prob_list, cycle_numbers).
    """
    ssn_list = []
    prob_list = []
    num_list = []

    for i, sc in enumerate(SOLAR_CYCLES):
        next_min = (SOLAR_CYCLES[i + 1].min_year
                    if i + 1 < len(SOLAR_CYCLES) else SC25_START)
        mask = (years >= sc.min_year) & (years < next_min)
        if mask.sum() == 0:
            continue
        n_storms = np.sum(aah[mask] > threshold)
        prob = n_storms / mask.sum()
        ssn_list.append(sc.mean_ssn)
        prob_list.append(prob)
        num_list.append(sc.number)

    return ssn_list, prob_list, num_list


# Scatter plots: cycle amplitude vs storm probability — cf. Figure 5
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, p in enumerate(percentiles):
    ax = axes[idx]
    ssn, prob, nums = cycle_storm_probability(thresholds[p])

    ax.scatter(ssn, prob, color="black", s=50, zorder=5)

    # Label each point with cycle number
    for s, pr, n in zip(ssn, prob, nums):
        ax.annotate(f"{n}", (s, pr), fontsize=7,
                    textcoords="offset points", xytext=(5, 5))

    # Linear fit and correlation
    ssn_arr = np.array(ssn)
    prob_arr = np.array(prob)
    r, p_val = stats.pearsonr(ssn_arr, prob_arr)

    # Regression line
    slope, intercept = np.polyfit(ssn_arr, prob_arr, 1)
    x_fit = np.linspace(ssn_arr.min(), ssn_arr.max(), 100)
    ax.plot(x_fit, slope * x_fit + intercept, "r--", alpha=0.7)

    ax.set_title(f"{p}th percentile\n"
                 f"N={int(np.sum(aah > thresholds[p]))} | "
                 f"r={r:.2f} (p={p_val:.4f})")
    ax.set_xlabel("Cycle-average SSN")
    ax.set_ylabel("Storm probability [day$^{-1}$]")

fig.suptitle("Cycle Amplitude vs Storm Probability / "
             "주기 진폭 vs 폭풍 확률 (cf. Figure 5)",
             fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation coefficient as function of threshold — cf. Figure 6
threshold_range = np.linspace(
    np.percentile(aah, 80), np.percentile(aah, 99.95), 50)

r_values = []
for t in threshold_range:
    ssn, prob, _ = cycle_storm_probability(t)
    if len(ssn) >= 3 and np.std(prob) > 0:
        r, _ = stats.pearsonr(ssn, prob)
        r_values.append(r)
    else:
        r_values.append(np.nan)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(threshold_range, r_values, "ko-", markersize=4)
ax.set_xlabel("$aa_H$ threshold [nT]")
ax.set_ylabel("Correlation $r$ (cycle amplitude vs storm probability)")
ax.set_title("Correlation vs Threshold / 상관계수 vs 임계값 (cf. Figure 6)")

# Mark the four percentile thresholds
for p, color in zip(percentiles, ["blue", "orange", "green", "red"]):
    t = thresholds[p]
    ax.axvline(t, color=color, ls="--", alpha=0.5, label=f"{p}th ({t:.0f} nT)")

ax.legend(fontsize=9)
ax.set_ylim(-0.5, 1.1)
ax.axhline(0, color="gray", ls=":", alpha=0.5)
plt.tight_layout()
plt.show()

print("Key insight: correlation peaks near the 99th percentile and declines")
print("at higher thresholds — but this is a sample-size effect, not a real")
print("weakening of the underlying relationship (Phase+Amp model reproduces this).")

## 6. Odd/Even Cycle Asymmetry / 홀수/짝수 주기 비대칭

The most novel finding: extreme events cluster in the early active phase of
even-numbered cycles and the late active phase of odd-numbered cycles.
This reproduces the analysis of **Figure 7** and the $p = 0.5^6 = 0.016$ test.

가장 새로운 발견: 극한 사건이 짝수 주기의 초기 활동기와 홀수 주기의 후기 활동기에
집중됩니다. **Figure 7**의 분석과 $p = 0.5^6 = 0.016$ 검정을 재현합니다.

In [ ]:
def classify_storms_early_late(
    storm_years: np.ndarray,
) -> dict[str, dict[str, int]]:
    """Classify storms by odd/even cycle and early/late active phase.

    Args:
        storm_years: Array of storm occurrence years.

    Returns:
        Dict with keys 'even'/'odd', each containing
        {'early': count, 'late': count, 'quiet': count}.
    """
    result = {
        "even": {"early": 0, "late": 0, "quiet": 0},
        "odd": {"early": 0, "late": 0, "quiet": 0},
    }

    for y in storm_years:
        sc = get_cycle_for_year(y)
        phase = get_phase(y)
        if sc is None or phase is None:
            continue

        parity = "even" if sc.is_even else "odd"
        if not is_active(phase):
            result[parity]["quiet"] += 1
        elif is_early_active(phase):
            result[parity]["early"] += 1
        else:
            result[parity]["late"] += 1

    return result


# Analyze at different thresholds
print("Storm classification by cycle parity and active-phase timing:")
print("=" * 70)

for p in percentiles:
    t = thresholds[p]
    storm_yrs = years[aah > t]
    counts = classify_storms_early_late(storm_yrs)

    print(f"\n{p}th percentile (>{t:.1f} nT, N={len(storm_yrs)}):")
    for parity in ["even", "odd"]:
        c = counts[parity]
        total_active = c["early"] + c["late"]
        early_frac = c["early"] / total_active if total_active > 0 else 0
        print(f"  {parity:>4} cycles: early={c['early']:>3}, "
              f"late={c['late']:>3}, quiet={c['quiet']:>3}  "
              f"(early fraction: {early_frac:.2f})")

In [ ]:
# Binomial test for odd/even asymmetry — reproduces the p = 0.5^6 calculation
# At the highest threshold, check if even-cycle storms favor early
# and odd-cycle storms favor late active phase

t_extreme = thresholds[99.99]
storm_yrs_extreme = years[aah > t_extreme]
counts_extreme = classify_storms_early_late(storm_yrs_extreme)

print("Odd/Even asymmetry test at 99.99th percentile:")
print("=" * 55)

n_even_early = counts_extreme["even"]["early"]
n_even_late = counts_extreme["even"]["late"]
n_odd_early = counts_extreme["odd"]["early"]
n_odd_late = counts_extreme["odd"]["late"]

n_even_total = n_even_early + n_even_late
n_odd_total = n_odd_early + n_odd_late

print(f"Even cycles: {n_even_early} early, {n_even_late} late "
      f"(of {n_even_total} active-phase storms)")
print(f"Odd cycles:  {n_odd_early} early, {n_odd_late} late "
      f"(of {n_odd_total} active-phase storms)")

# Under null hypothesis: each storm has 50/50 chance of early/late
# P(all even early AND all odd late) = 0.5^(n_even + n_odd)
n_consistent = n_even_early + n_odd_late  # storms matching the pattern
n_total_active = n_even_total + n_odd_total
if n_total_active > 0:
    p_chance = 0.5 ** n_total_active
    print(f"\nIf each storm has 50:50 early/late probability:")
    print(f"  P(observed pattern by chance) = 0.5^{n_total_active} "
          f"= {p_chance:.6f}")
    print(f"  = {p_chance * 100:.2f}%")
    if p_chance < 0.05:
        print("  → Statistically significant at 95% confidence level!")
    else:
        print("  → Not statistically significant (synthetic data has "
              "fewer extreme events than real data)")

## 7. Solar Cycle 25 Forecast / Solar Cycle 25 예측

Apply the three rules (phase, amplitude, odd/even) to forecast extreme-event
probability for SC25 under three scenarios. This reproduces **Figure 8**.

세 가지 규칙(위상, 진폭, 홀수/짝수)을 적용하여 SC25의 극한 사건 확률을
세 가지 시나리오로 예측합니다. **Figure 8**을 재현합니다.

SC25 scenarios:
- **Small** (SC12-class): mean SSN ≈ 64
- **Moderate** (SC23-class): mean SSN ≈ 80
- **Large** (SC19-class): mean SSN ≈ 126

In [ ]:
def sc25_probability(
    sc25_years: np.ndarray,
    mean_ssn: float,
    active_ratio: float = 9.0,
    amp_scale: float = 1.5,
    asymmetry: float = 0.6,
) -> np.ndarray:
    """Compute relative storm probability for SC25 using the EarlyLate model.

    SC25 is odd-numbered, so extreme events favor the late active phase.

    Args:
        sc25_years: Array of years within SC25.
        mean_ssn: Assumed mean sunspot number for SC25.
        active_ratio: Active/quiet probability ratio.
        amp_scale: Amplitude scaling factor.
        asymmetry: Early/late asymmetry magnitude.

    Returns:
        Array of relative probabilities (not normalized).
    """
    sc25_start = 2020.0
    sc25_end = 2031.0  # assumed ~11-year cycle
    probs = np.ones_like(sc25_years)

    for i, y in enumerate(sc25_years):
        phase = (y - sc25_start) / (sc25_end - sc25_start)
        if phase < 0 or phase > 1:
            continue

        if is_active(phase):
            amp_factor = amp_scale * mean_ssn / MEAN_SSN
            base = active_ratio * amp_factor

            # SC25 is odd: early gets reduced, late gets boosted
            if is_early_active(phase):
                probs[i] = base * (1 - asymmetry)
            else:
                probs[i] = base * (1 + asymmetry)

    return probs


# SC25 scenarios
sc25_years = np.linspace(2020, 2032, 365 * 12)
scenarios = {
    "Small (SC12)": {"ssn": 64, "color": "red", "ls": "-", "lw": 2.5},
    "Moderate (SC23)": {"ssn": 80, "color": "black", "ls": "--", "lw": 2},
    "Large (SC19)": {"ssn": 126, "color": "blue", "ls": "-", "lw": 1.5},
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: sunspot number proxy (simple sinusoidal model)
ax = axes[0]
for name, params in scenarios.items():
    # Simple sinusoidal SSN model peaking at phase ~0.35
    phase = (sc25_years - 2020) / 11
    ssn_model = params["ssn"] * 2.5 * np.maximum(
        0, np.sin(np.pi * phase) ** 1.5 * np.exp(-0.5 * (phase - 0.35)**2 / 0.1))
    ax.plot(sc25_years, ssn_model, color=params["color"],
            ls=params["ls"], lw=params["lw"], label=name)
ax.set_ylabel("Sunspot Number")
ax.set_title("SSN Scenarios / 흑점 수 시나리오")
ax.legend(fontsize=8)
ax.set_xlim(2020, 2032)

# Middle: 99th percentile storm probability
ax = axes[1]
for name, params in scenarios.items():
    probs = sc25_probability(sc25_years, params["ssn"])
    # Smooth and normalize to approximate daily probability
    probs_smooth = np.convolve(probs, np.ones(90) / 90, mode="same")
    probs_norm = probs_smooth / probs_smooth.sum() * 548 / 365  # scale to ~99th pct
    ax.plot(sc25_years, probs_norm, color=params["color"],
            ls=params["ls"], lw=params["lw"], label=name)
ax.set_ylabel("Probability [day$^{-1}$]")
ax.set_title("99th pctile ($aa_H$ > 77 nT)\n\"Moderate\" storms")
ax.set_xlim(2020, 2032)

# Right: 99.99th percentile storm probability
ax = axes[2]
for name, params in scenarios.items():
    probs = sc25_probability(sc25_years, params["ssn"])
    probs_smooth = np.convolve(probs, np.ones(90) / 90, mode="same")
    probs_norm = probs_smooth / probs_smooth.sum() * 6 / 365  # scale to ~99.99th pct
    ax.plot(sc25_years, probs_norm, color=params["color"],
            ls=params["ls"], lw=params["lw"], label=name)
ax.set_ylabel("Probability [day$^{-1}$]")
ax.set_title("99.99th pctile ($aa_H$ > 290 nT)\n\"Extreme\" storms")
ax.set_xlim(2020, 2032)

fig.suptitle("Solar Cycle 25 Storm Probability Forecast / "
             "SC25 폭풍 확률 예측 (cf. Figure 8)", fontsize=13)
plt.tight_layout()
plt.show()

# Highlight the key prediction
print("Key prediction for SC25 (odd cycle):")
print("  → Extreme events (99.99th percentile) peak in the LATE active phase")
print("    (expected after ~2026), consistent with odd-cycle behavior.")
print("  → For a large cycle (SC19-class), integrated probability of at least")
print("    one extreme event over 11 years is ~54%; for a small cycle, ~24%.")

## 8. Summary / 요약

### What we implemented / 구현한 내용

1. **Synthetic $aa_H$ generation**: Created a 150-year daily geomagnetic record with solar-cycle modulation, amplitude dependence, and odd/even asymmetry built in.  
   150년 일별 지자기 기록을 태양 주기 조절, 진폭 의존성, 홀수/짝수 비대칭을 포함하여 생성.

2. **Four probability models**: Hierarchical models (Random → Phase → Phase+Amp → EarlyLate), each implemented as a class with inverse-CDF storm generation.  
   계층적 확률 모델 4가지를 클래스로 구현하고, 역-CDF 방식의 폭풍 생성 기능 포함.

3. **Monte Carlo testing (cf. Figure 4)**: 1000-iteration simulation showing that the Random model (null hypothesis) is rejected at the 2σ level for active/quiet storm probability differences.  
   1000회 반복 시뮬레이션으로 Random 모델(귀무가설)이 2σ 수준에서 기각됨을 확인.

4. **Cycle-amplitude correlation (cf. Figures 5–6)**: Scatter plots and correlation analysis confirming that bigger cycles produce more extreme storms, with $r$ peaking near the 99th percentile.  
   산점도와 상관 분석으로 큰 주기가 더 많은 극한 폭풍을 생산함을 확인. $r$이 99th percentile 근처에서 피크.

5. **Odd/even asymmetry (cf. Figure 7)**: Classification showing even-cycle storms favor early active phase and odd-cycle storms favor late active phase, with binomial significance test.  
   짝수 주기 폭풍은 초기 활동기를, 홀수 주기 폭풍은 후기 활동기를 선호함을 이항 유의성 검정으로 확인.

6. **SC25 forecast (cf. Figure 8)**: Three-scenario probability forecast showing extreme events peaking in the late active phase (after ~2026) for odd-numbered SC25.  
   SC25(홀수 주기)의 극한 사건이 후기 활동기(~2026년 이후)에 피크를 보이는 세 가지 시나리오 예측.

### Key takeaway / 핵심 교훈

The paper's methodology demonstrates that even with very few extreme events (6 storms at the 99.99th percentile over 150 years), probabilistic models combined with Monte Carlo sampling can extract statistically significant conclusions about solar-cycle dependence.

이 논문의 방법론은 매우 적은 수의 극한 사건(150년간 99.99th percentile에서 6개 폭풍)으로도 확률론적 모델과 Monte Carlo 샘플링을 결합하면 태양 주기 의존성에 대한 통계적으로 유의미한 결론을 도출할 수 있음을 보여줍니다.